In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers datasets accelerate librosa soundfile evaluate jiwer
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
print('✅ All dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.3 MB/s eta 0:00:00
✅ All dependencies installed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ✏️ UPDATE THIS PATH to your folder containing audio_*.wav and audio_*.txt files
DATASET_DIR = '/content/drive/MyDrive/dataset_k'

# Output directory for the fine-tuned model
OUTPUT_DIR = '/content/drive/MyDrive/whisper_tiny_finetuned'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Dataset path: {DATASET_DIR}')
print(f'Output path:  {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset path: /content/drive/MyDrive/dataset_k
Output path:  /content/drive/MyDrive/whisper_tiny_finetuned


In [ ]:
import librosa
import soundfile as sf
import numpy as np
from pathlib import Path

SAMPLE_RATE = 16000  # Whisper requires 16kHz

data_records = []
skipped = []

dataset_path = Path(DATASET_DIR)
wav_files = sorted(dataset_path.glob('audio_*.wav'))

print(f'Found {len(wav_files)} .wav files.')

for wav_path in wav_files:
    txt_path = wav_path.with_suffix('.txt')

    # Check transcript exists
    if not txt_path.exists():
        skipped.append((wav_path.name, 'Missing .txt'))
        continue

    # Read transcript
    transcript = txt_path.read_text(encoding='utf-8').strip()
    if not transcript:
        skipped.append((wav_path.name, 'Empty transcript'))
        continue

    # Load and validate audio
    try:
        audio, sr = librosa.load(str(wav_path), sr=SAMPLE_RATE, mono=True)
        duration = len(audio) / SAMPLE_RATE

        # Skip very short (<0.5s) or very long (>30s) clips
        if duration < 0.5 or duration > 30.0:
            skipped.append((wav_path.name, f'Duration {duration:.1f}s out of range'))
            continue

        data_records.append({
            'audio_path': str(wav_path),
            'transcript': transcript,
            'duration': duration
        })
    except Exception as e:
        skipped.append((wav_path.name, str(e)))

print(f'\n✅ Valid samples : {len(data_records)}')
print(f'⚠️  Skipped       : {len(skipped)}')
if skipped:
    for name, reason in skipped:
        print(f'   {name}: {reason}')

durations = [r['duration'] for r in data_records]
print(f'\n📊 Duration stats:')
print(f'   Min: {min(durations):.1f}s | Max: {max(durations):.1f}s | Avg: {np.mean(durations):.1f}s')

Found 206 .wav files.

✅ Valid samples : 206
⚠️  Skipped       : 0

📊 Duration stats:
   Min: 4.9s | Max: 13.2s | Avg: 7.7s


In [ ]:
from datasets import Dataset, Audio
import pandas as pd

df = pd.DataFrame(data_records)
hf_dataset = Dataset.from_pandas(df[['audio_path', 'transcript']])

# Cast audio column so HuggingFace handles loading
hf_dataset = hf_dataset.rename_column('audio_path', 'audio')
hf_dataset = hf_dataset.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

# 90/10 train-validation split
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
val_dataset   = split['test']

print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')

Train samples : 185
Val samples   : 21


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = 'openai/whisper-tiny'

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Force English language & transcription task
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='english', task='transcribe'
)
model.config.suppress_tokens = []

print('✅ Model and processor loaded.')

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

✅ Model and processor loaded.


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

def prepare_dataset(batch):
    audio = batch['audio']
    # Extract log-mel spectrogram
    batch['input_features'] = processor.feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate']
    ).input_features[0]
    # Tokenize transcript
    batch['labels'] = processor.tokenizer(batch['transcript']).input_ids
    return batch

print('Preprocessing training set...')
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc=1)
print('Preprocessing validation set...')
val_dataset   = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names, num_proc=1)
print('✅ Preprocessing done.')

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Remove BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('✅ Data collator ready.')

Preprocessing training set...


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

Preprocessing validation set...


Map:   0%|          | 0/21 [00:00<?, ? examples/s]

✅ Preprocessing done.
✅ Data collator ready.


In [ ]:
import evaluate

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding in labels
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

print('✅ WER metric ready.')

✅ WER metric ready.


In [ ]:
dataset_path = "/content/drive/MyDrive/dataset_k"

In [ ]:
import os

print(os.listdir(dataset_path))

['audio_1.txt', 'audio_1.wav', 'audio_2.wav', 'audio_2.txt', 'audio_3.wav', 'audio_3.txt', 'audio_4.txt', 'audio_4.wav', 'audio_5.txt', 'audio_5.wav', 'audio_6.wav', 'audio_6.txt', 'audio_7.wav', 'audio_7.txt', 'audio_8.wav', 'audio_8.txt', 'audio_9.wav', 'audio_9.txt', 'audio_10.txt', 'audio_10.wav', 'audio_11.wav', 'audio_11.txt', 'audio_12.txt', 'audio_12.wav', 'audio_13.wav', 'audio_13.txt', 'audio_14.wav', 'audio_14.txt', 'audio_15.txt', 'audio_15.wav', 'audio_16.wav', 'audio_16.txt', 'audio_17.wav', 'audio_17.txt', 'audio_18.wav', 'audio_18.txt', 'audio_19.wav', 'audio_19.txt', 'audio_20.wav', 'audio_20.txt', 'audio_21.wav', 'audio_21.txt', 'audio_22.wav', 'audio_22.txt', 'audio_23.txt', 'audio_23.wav', 'audio_24.wav', 'audio_24.txt', 'audio_25.wav', 'audio_25.txt', 'audio_26.txt', 'audio_26.wav', 'audio_27.wav', 'audio_27.txt', 'audio_28.wav', 'audio_28.txt', 'audio_29.wav', 'audio_29.txt', 'audio_30.wav', 'audio_30.txt', 'audio_31.wav', 'audio_31.txt', 'audio_32.wav', 'audio_32

In [ ]:
# ============================================================
# Fine-tune Whisper Tiny on Custom English Audio
# Full pipeline: Preprocess → Train → Evaluate → Save
# ============================================================
# Requirements:
#   audio_1.wav ... audio_206.wav
#   audio_1.txt ... audio_206.txt  (plain text transcripts)
#
# Run on Google Colab with T4 GPU
# ============================================================

# ── Step 1: Install Dependencies ────────────────────────────
# Run this in a Colab cell before executing the script:
# !pip install -q transformers datasets accelerate librosa soundfile evaluate jiwer

# ── Step 2: Imports ─────────────────────────────────────────
import os
import numpy as np
import librosa
import torch
import evaluate
import pandas as pd

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline,
)
from google.colab import drive

# ── Step 3: Mount Drive & Config ────────────────────────────
drive.mount('/content/drive')

# ✏️ UPDATE these paths
DATASET_DIR = '/content/drive/MyDrive/dataset_k'
OUTPUT_DIR  = '/content/drive/MyDrive/whisper_tiny_finetuned'

SAMPLE_RATE = 16000
MODEL_NAME  = 'openai/whisper-tiny'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Dataset : {DATASET_DIR}')
print(f'Output  : {OUTPUT_DIR}')

# ── Step 4: Validate Dataset ────────────────────────────────
data_records = []
skipped = []

dataset_path = Path(DATASET_DIR)
wav_files = sorted(dataset_path.glob('audio_*.wav'))
print(f'\nFound {len(wav_files)} .wav files.')

for wav_path in wav_files:
    txt_path = wav_path.with_suffix('.txt')

    if not txt_path.exists():
        skipped.append((wav_path.name, 'Missing .txt'))
        continue

    transcript = txt_path.read_text(encoding='utf-8').strip()
    if not transcript:
        skipped.append((wav_path.name, 'Empty transcript'))
        continue

    try:
        audio, sr = librosa.load(str(wav_path), sr=SAMPLE_RATE, mono=True)
        duration = len(audio) / SAMPLE_RATE

        if duration < 0.5 or duration > 30.0:
            skipped.append((wav_path.name, f'Duration {duration:.1f}s out of range'))
            continue

        data_records.append({
            'audio_path': str(wav_path),
            'transcript': transcript,
            'duration': duration
        })
    except Exception as e:
        skipped.append((wav_path.name, str(e)))

print(f'✅ Valid   : {len(data_records)}')
print(f'⚠️  Skipped : {len(skipped)}')
for name, reason in skipped:
    print(f'   {name}: {reason}')

durations = [r['duration'] for r in data_records]
print(f'\nDuration — Min: {min(durations):.1f}s | Max: {max(durations):.1f}s | Avg: {np.mean(durations):.1f}s')

# ── Step 5: Build HuggingFace Dataset ───────────────────────
df = pd.DataFrame(data_records)
hf_dataset = Dataset.from_pandas(df[['audio_path', 'transcript']])
hf_dataset = hf_dataset.rename_column('audio_path', 'audio')
hf_dataset = hf_dataset.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
val_dataset   = split['test']

print(f'\nTrain : {len(train_dataset)} | Val : {len(val_dataset)}')

# ── Step 6: Load Model & Processor ──────────────────────────
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='english', task='transcribe'
)
model.generation_config.suppress_tokens = []
print('✅ Model loaded.')

# ── Step 7: Feature Extraction ──────────────────────────────
def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor.feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['transcript']).input_ids
    return batch

print('Preprocessing...')
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc=1)
val_dataset   = val_dataset.map(prepare_dataset,   remove_columns=val_dataset.column_names,   num_proc=1)
print('✅ Preprocessing done.')

# ── Step 8: Data Collator ───────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ── Step 9: Metric ──────────────────────────────────────────
wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

# ── Step 10: Training ───────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training schedule
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    warmup_steps=50,
    learning_rate=1e-5,
    weight_decay=0.01,

    # Precision & generation
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,

    # Eval & saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    save_total_limit=2,

    # Logging
    logging_steps=10,
    report_to='none',
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

# ── Step 11: Save ───────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f'✅ Model saved to: {OUTPUT_DIR}')

# ── Step 12: Inference Test ─────────────────────────────────
TEST_AUDIO = str(sorted(Path(DATASET_DIR).glob('audio_*.wav'))[0])

asr = pipeline(
    task='automatic-speech-recognition',
    model=OUTPUT_DIR,
    chunk_length_s=30,
    device=0
)

result = asr(TEST_AUDIO)
gt     = Path(TEST_AUDIO).with_suffix('.txt').read_text().strip()

print(f'\nAudio        : {TEST_AUDIO}')
print(f'Transcription: {result["text"]}')
print(f'Ground truth : {gt}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset : /content/drive/MyDrive/dataset_k
Output  : /content/drive/MyDrive/whisper_tiny_finetuned

Found 206 .wav files.
✅ Valid   : 206
⚠️  Skipped : 0

Duration — Min: 4.9s | Max: 13.2s | Avg: 7.7s

Train : 185 | Val : 21


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

✅ Model loaded.
Preprocessing...


Map:   0%|          | 0/185 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: /content/drive/MyDrive/whisper_tiny_finetuned


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).



Audio        : /content/drive/MyDrive/dataset_k/audio_1.wav
Transcription: virat kohli is studying data science at goa university while playing cricket
Ground truth : virat kohli is studying data science at goa university while playing cricket


In [ ]:
!cp -r "/content/drive/MyDrive/dataset_k" /content/
dataset_path = "/content/dataset_k"

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/dataset_k"

data = []

for file in os.listdir(dataset_path):
    if file.endswith(".wav"):
        audio_file = os.path.join(dataset_path, file)
        text_file = audio_file.replace(".wav", ".txt")

        if os.path.exists(text_file):
            with open(text_file, "r", encoding="utf-8") as f:
                text = f.read().strip()

            data.append({
                "audio": audio_file,
                "text": text
            })

print("Total samples:", len(data))

Total samples: 206


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(data)

In [ ]:
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# verify
print(dataset[0]["audio"])

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
def preprocess(batch):
    audio = batch["audio"]

    # input features
    inputs = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    )

    # labels
    labels = processor.tokenizer(batch["text"]).input_ids

    return {
        "input_features": inputs.input_features[0],
        "labels": labels
    }

dataset = dataset.map(preprocess)

Map:   0%|          | 0/206 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2Seq(processor=processor)

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

# Important settings
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=True,

    do_eval=True,   # ✅ instead of evaluation_strategy

    per_device_eval_batch_size=8,
    save_steps=200,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    remove_unused_columns=False,
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()